In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
def qft(wires):
    n = len(wires)
    for i in range(n):
        qml.Hadamard(wires[i])
        for j in range(i+1, n):
            angle = np.pi/(2**(j-i))
            qml.ControlledPhaseShift(angle, wires=[wires[j], wires[i]])
    for i in range(n//2):
        qml.SWAP(wires=[wires[i], wires[n-i-1]])

def inverse_qft(wires):
    qml.adjoint(qft)(wires)

In [3]:
dev = qml.device('default.qubit', wires=3)

@qml.qnode(dev)
def test_qft():
    qml.BasisState(np.array([1, 0, 1]), wires=range(3))
    qft(wires=range(3))
    return qml.state()

state = test_qft()
print("QFT of |101>:\n", state)   

QFT of |101>:
 [ 3.53553391e-01+0.j         -2.50000000e-01-0.25j
  2.16489014e-17+0.35355339j  2.50000000e-01-0.25j
 -3.53553391e-01+0.j          2.50000000e-01+0.25j
 -2.16489014e-17-0.35355339j -2.50000000e-01+0.25j      ]


In [4]:
n_ancilla = 3
n_system = 1

wires = range(n_ancilla + n_system)

dev = qml.device('default.qubit', wires=wires)

ground_truth = 3/8

@qml.qnode(dev)
def qpe_circ():
    for i in range(n_ancilla):
        qml.Hadamard(wires=i)
    qml.PauliX(wires=n_ancilla)

    for k in range(n_ancilla):
        power = 2**(n_ancilla-k-1)
        angle = 2*np.pi*ground_truth*power
        qml.ControlledPhaseShift(angle, wires = [k, n_ancilla])

    inverse_qft(wires=range(n_ancilla))

    return qml.probs(wires=range(n_ancilla))

probs = qpe_circ()
estimated_phase = np.argmax(probs) / (2**n_ancilla)

print(f'True phase: {ground_truth}')
print(f'Estimated phase: {estimated_phase}')
print(f'Error = {abs(1-estimated_phase/ground_truth)}')

True phase: 0.375
Estimated phase: 0.375
Error = 0.0
